# Heathrow Passenger, Revenue and Cost Forecasting Model
## 01 — Data Collection and Audit

### Purpose
Collect and validate published Heathrow passenger and financial data.

### Data principles
- Keep original source files unchanged.
- Record source URLs, reporting periods and units.
- Distinguish reported data from model assumptions and forecast outputs.
- Do not present modelled monthly financial figures as Heathrow's actual accounts.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project folder:", PROJECT_ROOT)
print("Raw data folder exists:", RAW_DIR.is_dir())

Project folder: /Users/dafyddjones/projects/heathrow-forecasting-model
Raw data folder exists: True


In [2]:
raw_files = sorted(
    path.name
    for path in RAW_DIR.iterdir()
    if path.is_file() and not path.name.startswith(".")
)

raw_files

['sep-2026-traffic-statistics.xlsx']

In [3]:
traffic_path = RAW_DIR / "sep-2026-traffic-statistics.xlsx"

traffic_excel = pd.ExcelFile(traffic_path)

traffic_excel.sheet_names

['Heathrow Traffic Statistics',
 'Passengers by Market',
 'Cargo by Market',
 'Cargo+Mail by Market']

In [4]:
traffic_preview = pd.read_excel(
    traffic_excel,
    sheet_name="Heathrow Traffic Statistics",
    header=None
)

traffic_preview.head(15)

,0,1,2,3
0,Heathrow Traffic Statistics,NaN,NaN,NaN
1,January 2005 - September 2026,NaN,NaN,NaN
2,Month,Passengers,Air Transport Movements,Cargo\n(Metric Tonnes)
3,2005-01-01 00:00:00,5141123,39139,98781.175
4,2005-02-01 00:00:00,4753591,35723,99555.454
5,2005-03-01 00:00:00,5708627,39634,109387.896
6,2005-04-01 00:00:00,5573022,39096,108057.553
7,2005-05-01 00:00:00,5636621,40432,110612.691
8,2005-06-01 00:00:00,6000208,39640,109860.506
9,2005-07-01 00:00:00,6456943,41049,110844.073


In [5]:
traffic = pd.read_excel(
    traffic_excel,
    sheet_name="Heathrow Traffic Statistics",
    header=2
)

traffic.columns = [
    "month",
    "passengers",
    "air_transport_movements",
    "cargo_tonnes"
]

traffic.head()

,month,passengers,air_transport_movements,cargo_tonnes
0,2005-01-01,5141123.0,39139.0,98781.175
1,2005-02-01,4753591.0,35723.0,99555.454
2,2005-03-01,5708627.0,39634.0,109387.896
3,2005-04-01,5573022.0,39096.0,108057.553
4,2005-05-01,5636621.0,40432.0,110612.691


In [6]:
traffic.info()

<class 'pandas.DataFrame'>
RangeIndex: 261 entries, 0 to 260
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   month                    261 non-null    datetime64[us]
 1   passengers               261 non-null    float64       
 2   air_transport_movements  261 non-null    float64       
 3   cargo_tonnes             261 non-null    float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 8.3 KB


In [7]:
print("First month:", traffic["month"].min())
print("Last month:", traffic["month"].max())
print("Duplicate months:", traffic["month"].duplicated().sum())

expected_months = pd.date_range(
    start=traffic["month"].min(),
    end=traffic["month"].max(),
    freq="MS"
)

missing_months = expected_months.difference(traffic["month"])

print("Missing months:", missing_months.tolist())

First month: 2005-01-01 00:00:00
Last month: 2026-09-01 00:00:00
Duplicate months: 0
Missing months: []


In [8]:
traffic_monthly = (
    traffic.loc[
        traffic["month"].between("2024-10-01", "2026-09-01")
    ]
    .sort_values("month")
    .reset_index(drop=True)
    .copy()
)

print("Rows:", len(traffic_monthly))
print("First month:", traffic_monthly["month"].min())
print("Last month:", traffic_monthly["month"].max())

Rows: 24
First month: 2024-10-01 00:00:00
Last month: 2026-09-01 00:00:00


In [9]:
count_columns = ["passengers", "air_transport_movements"]

print("Negative values:")
print(
    (traffic_monthly[
        count_columns + ["cargo_tonnes"]
    ] < 0).sum()
)

print("\nFractional passenger or movement counts:")
print(
    (traffic_monthly[count_columns] % 1 != 0).sum()
)

Negative values:
passengers                 0
air_transport_movements    0
cargo_tonnes               0
dtype: int64

Fractional passenger or movement counts:
passengers                 0
air_transport_movements    0
dtype: int64


In [10]:
traffic_monthly[count_columns] = (
    traffic_monthly[count_columns].astype("int64")
)

traffic_monthly.head()

,month,passengers,air_transport_movements,cargo_tonnes
0,2024-10-01,7214853,40980,140598.578
1,2024-11-01,6500620,38084,139204.437
2,2024-12-01,7074890,38854,132616.508
3,2025-01-01,6319076,38699,119329.942
4,2025-02-01,5709458,35806,120765.418


In [11]:
output_path = (
    PROJECT_ROOT / "data" / "processed" / "heathrow_traffic_monthly.csv"
)

traffic_monthly.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved:", output_path)
print("Rows:", len(traffic_monthly))

Saved: /Users/dafyddjones/projects/heathrow-forecasting-model/data/processed/heathrow_traffic_monthly.csv
Rows: 24


In [12]:
saved_traffic = pd.read_csv(output_path, parse_dates=["month"])

print("File exists:", output_path.is_file())
print("Rows and columns:", saved_traffic.shape)
print("Matches prepared data:", saved_traffic.equals(traffic_monthly))

File exists: True
Rows and columns: (24, 4)
Matches prepared data: False


In [13]:
comparison = pd.DataFrame({
    "original_dtype": traffic_monthly.dtypes.astype(str),
    "saved_dtype": saved_traffic.dtypes.astype(str),
    "values_match": [
        traffic_monthly[col].equals(saved_traffic[col])
        for col in traffic_monthly.columns
    ]
})

comparison

,original_dtype,saved_dtype,values_match
month,datetime64[us],datetime64[us],True
passengers,int64,int64,True
air_transport_movements,int64,int64,True
cargo_tonnes,float64,float64,False


In [14]:
cargo_difference = (
    traffic_monthly["cargo_tonnes"] - saved_traffic["cargo_tonnes"]
).abs()

print("Largest difference in tonnes:", cargo_difference.max())
print(
    "Matches within one millionth of a tonne:",
    np.allclose(
        traffic_monthly["cargo_tonnes"],
        saved_traffic["cargo_tonnes"],
        rtol=0,
        atol=0.000001
    )
)

Largest difference in tonnes: 1.4551915228366852e-11
Matches within one millionth of a tonne: True


### Traffic data audit

- Source file: sep-2026-traffic-statistics.xlsx
- Worksheet: Heathrow Traffic Statistics
- Full source coverage: January 2005–September 2026
- Selected project period: October 2024–September 2026 (24 months)
- No missing values, missing months or duplicate months.
- No negative values in the selected period.
- Passenger and aircraft-movement counts verified as whole numbers.
- Prepared CSV checked against the original table: dates and counts match
  exactly; cargo tonnage matches within numerical precision.
- Output: data/processed/heathrow_traffic_monthly.csv

In [15]:
market_preview = pd.read_excel(
    traffic_excel,
    sheet_name="Passengers by Market",
    header=None
)

market_preview.head(12)

,0,1,2,3,4,5,6,7,8,9
0,Passengers by Market,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,January 2005 - September 2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Month,UK,EU,Non-EU Europe,Africa,North America,Latin America,Middle East,Asia / Pacific,Total
3,2005-01-01 00:00:00,520535,1744522,296634,358609,1019101,84992,351909,764821,5141123
4,2005-02-01 00:00:00,499676,1750604,284788,306822,880957,69034,292150,669560,4753591
5,2005-03-01 00:00:00,588602,2090599,336179,345153,1151386,72028,334172,790508,5708627
6,2005-04-01 00:00:00,568420,2087496,321400,302564,1179010,70882,313878,729372,5573022
7,2005-05-01 00:00:00,559711,2198082,320871,276538,1224247,65004,290197,701971,5636621
8,2005-06-01 00:00:00,578167,2274306,346605,292420,1333320,68016,316121,791253,6000208
9,2005-07-01 00:00:00,591089,2432169,394557,348344,1368455,78329,379237,864763,6456943


In [16]:
passengers_by_market = pd.read_excel(
    traffic_excel,
    sheet_name="Passengers by Market",
    header=2
)

passengers_by_market.columns = [
    "month",
    "uk",
    "eu",
    "non_eu_europe",
    "africa",
    "north_america",
    "latin_america",
    "middle_east",
    "asia_pacific",
    "total_passengers"
]

passengers_by_market.head()

,month,uk,eu,non_eu_europe,africa,north_america,latin_america,middle_east,asia_pacific,total_passengers
0,2005-01-01,520535.0,1744522.0,296634.0,358609.0,1019101.0,84992.0,351909.0,764821.0,5141123.0
1,2005-02-01,499676.0,1750604.0,284788.0,306822.0,880957.0,69034.0,292150.0,669560.0,4753591.0
2,2005-03-01,588602.0,2090599.0,336179.0,345153.0,1151386.0,72028.0,334172.0,790508.0,5708627.0
3,2005-04-01,568420.0,2087496.0,321400.0,302564.0,1179010.0,70882.0,313878.0,729372.0,5573022.0
4,2005-05-01,559711.0,2198082.0,320871.0,276538.0,1224247.0,65004.0,290197.0,701971.0,5636621.0


In [17]:
market_monthly = (
    passengers_by_market.loc[
        passengers_by_market["month"].between(
            "2024-10-01", "2026-09-01"
        )
    ]
    .sort_values("month")
    .reset_index(drop=True)
    .copy()
)

print("Rows:", len(market_monthly))
print("Missing values:", market_monthly.isna().sum().sum())
print("Duplicate months:", market_monthly["month"].duplicated().sum())

Rows: 24
Missing values: 0
Duplicate months: 0


In [18]:
market_columns = [
    "uk", "eu", "non_eu_europe", "africa",
    "north_america", "latin_america",
    "middle_east", "asia_pacific"
]

regional_difference = (
    market_monthly[market_columns].sum(axis=1)
    - market_monthly["total_passengers"]
)

traffic_difference = (
    market_monthly.set_index("month")["total_passengers"]
    - traffic_monthly.set_index("month")["passengers"]
)

print("Largest regional total difference:", regional_difference.abs().max())
print("Largest traffic total difference:", traffic_difference.abs().max())

Largest regional total difference: 0.0
Largest traffic total difference: 0.0


In [19]:
market_count_columns = market_columns + ["total_passengers"]

print(
    "Negative values:",
    (market_monthly[market_count_columns] < 0).sum().sum()
)

print(
    "Fractional counts:",
    (market_monthly[market_count_columns] % 1 != 0).sum().sum()
)

Negative values: 0
Fractional counts: 0


In [23]:
market_monthly[market_count_columns] = (
    market_monthly[market_count_columns].astype("int64")
)

market_monthly.head()

,month,uk,eu,non_eu_europe,africa,north_america,latin_america,middle_east,asia_pacific,total_passengers
0,2024-10-01,393306,2505993,470047,271006,1851794,178801,683709,860197,7214853
1,2024-11-01,411802,2115248,411596,276893,1482146,198832,726568,877535,6500620
2,2024-12-01,400965,2228529,483479,297445,1700961,210844,801884,950783,7074890
3,2025-01-01,348480,1838699,452107,302179,1401440,216835,773976,985360,6319076
4,2025-02-01,334819,1782916,423306,271567,1161023,190636,686893,858298,5709458


In [25]:
market_output_path = (
    PROJECT_ROOT / "data" / "processed"
    / "heathrow_passengers_by_market_monthly.csv"
)

market_monthly.to_csv(
    market_output_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved:", market_output_path.name)
print("Rows:", len(market_monthly))

Saved: heathrow_passengers_by_market_monthly.csv
Rows: 24


### Passenger-market data audit

- Worksheet: Passengers by Market
- Selected period: October 2024–September 2026 (24 months)
- Eight market groupings retained as published by Heathrow.
- No missing values or duplicate months.
- All passenger counts are non-negative whole numbers.
- Regional passenger counts reconcile exactly to the worksheet total.
- Worksheet totals reconcile exactly to the main traffic table.
- Output: data/processed/heathrow_passengers_by_market_monthly.csv

### Traffic-data source

- Publisher: Heathrow
- Source page: https://www.heathrow.com/company/investor-centre/reports/traffic-statistics
- Downloaded file: sep-2026-traffic-statistics.xlsx
- Worksheets used: Heathrow Traffic Statistics; Passengers by Market
- Source coverage: January 2005–September 2026
- Model period selected: October 2024–September 2026
- Units: passengers and aircraft movements are counts; cargo is metric tonnes.
- Original workbook retained unchanged in data/raw/.

In [26]:
pdf_files = sorted(
    path.name
    for path in RAW_DIR.iterdir()
    if path.suffix.lower() == ".pdf"
)

pdf_files

['Heathrow_(SP)_Limited_FY_2025.pdf']

In [27]:
financial_annual = pd.DataFrame({
    "year": [2024, 2025],
    "revenue_gbp_m": [3559, 3623],
    "adjusted_operating_costs_gbp_m": [1524, 1589],
    "adjusted_ebitda_gbp_m": [2035, 2034]
})

financial_annual

,year,revenue_gbp_m,adjusted_operating_costs_gbp_m,adjusted_ebitda_gbp_m
0,2024,3559,1524,2035
1,2025,3623,1589,2034


In [28]:
financial_annual["ebitda_check_gbp_m"] = (
    financial_annual["revenue_gbp_m"]
    - financial_annual["adjusted_operating_costs_gbp_m"]
    - financial_annual["adjusted_ebitda_gbp_m"]
)

financial_annual[["year", "ebitda_check_gbp_m"]]

,year,ebitda_check_gbp_m
0,2024,0
1,2025,0


In [29]:
financial_annual["reporting_entity"] = "Heathrow (SP) Limited"
financial_annual["data_status"] = "reported_actual"
financial_annual["source_file"] = "Heathrow_(SP)_Limited_FY_2025.pdf"

financial_annual

,year,revenue_gbp_m,adjusted_operating_costs_gbp_m,adjusted_ebitda_gbp_m,ebitda_check_gbp_m,reporting_entity,data_status,source_file
0,2024,3559,1524,2035,0,Heathrow (SP) Limited,reported_actual,Heathrow_(SP)_Limited_FY_2025.pdf
1,2025,3623,1589,2034,0,Heathrow (SP) Limited,reported_actual,Heathrow_(SP)_Limited_FY_2025.pdf


In [30]:
financial_output_path = (
    PROJECT_ROOT / "data" / "processed"
    / "heathrow_financial_annual.csv"
)

financial_annual.to_csv(financial_output_path, index=False)

print("Saved:", financial_output_path.name)
print("Rows:", len(financial_annual))

Saved: heathrow_financial_annual.csv
Rows: 2


In [31]:
annual_passengers = (
    traffic.loc[
        traffic["month"].between("2024-01-01", "2025-12-01")
    ]
    .assign(year=lambda df: df["month"].dt.year)
    .groupby("year", as_index=False)
    .agg(
        months=("month", "nunique"),
        passengers=("passengers", "sum")
    )
)

annual_passengers

,year,months,passengers
0,2024,12,83859729.0
1,2025,12,84463061.0


In [32]:
financial_annual = financial_annual.merge(
    annual_passengers,
    on="year",
    how="left",
    validate="one_to_one"
)

financial_annual[
    ["year", "months", "passengers", "revenue_gbp_m"]
]

,year,months,passengers,revenue_gbp_m
0,2024,12,83859729.0,3559
1,2025,12,84463061.0,3623


In [33]:
financial_annual["revenue_per_passenger_gbp"] = (
    financial_annual["revenue_gbp_m"] * 1_000_000
    / financial_annual["passengers"]
)

financial_annual[
    ["year", "revenue_per_passenger_gbp"]
].round(2)

,year,revenue_per_passenger_gbp
0,2024,42.44
1,2025,42.89


In [34]:
financial_annual["operating_cost_per_passenger_gbp"] = (
    financial_annual["adjusted_operating_costs_gbp_m"] * 1_000_000
    / financial_annual["passengers"]
)

financial_annual[
    ["year", "operating_cost_per_passenger_gbp"]
].round(2)

,year,operating_cost_per_passenger_gbp
0,2024,18.17
1,2025,18.81


In [35]:
financial_annual["adjusted_ebitda_margin_pct"] = (
    financial_annual["adjusted_ebitda_gbp_m"]
    / financial_annual["revenue_gbp_m"]
    * 100
)

financial_annual[
    ["year", "adjusted_ebitda_margin_pct"]
].round(2)

,year,adjusted_ebitda_margin_pct
0,2024,57.18
1,2025,56.14


In [36]:
financial_annual.to_csv(financial_output_path, index=False)

print("Saved:", financial_output_path.name)
print("Rows:", len(financial_annual))

Saved: heathrow_financial_annual.csv
Rows: 2


### Annual financial benchmarks

- Reporting entity: Heathrow (SP) Limited.
- Source: Heathrow_(SP)_Limited_FY_2025.pdf.
- Periods: full calendar years 2024 and 2025.
- Financial amounts are reported in £ million.
- Annual passenger totals use all 12 months of each matching year.
- Revenue less adjusted operating costs reconciles to adjusted EBITDA.

| Measure | 2024 | 2025 |
|---|---:|---:|
| Total revenue per passenger | £42.44 | £42.89 |
| Adjusted operating cost per passenger | £18.17 | £18.81 |
| Adjusted EBITDA margin | 57.18% | 56.14% |

Per-passenger figures are calculated averages, not marginal prices or costs.
These annual benchmarks are not reported monthly financial results.